# Ch.4 — Diffusion Models Today: Score SDE Framework

**Goal:** turn the discrete noise ladders from DDPM (Ch. 2) and NCSN (Ch. 3) into one
continuous-time framework, then train one time-dependent score model and sample it with
multiple numerical solvers.

**Structure**
1. From discrete noise levels to a continuous-time SDE
2. A unified `SDE` interface for VE-SDE (NCSN) and VP-SDE (DDPM)
3. Continuous-time denoising score matching
4. Reverse-time SDE sampling with Euler–Maruyama
5. Probability Flow ODE (PF-ODE) sampling with RK4
6. VP/DDPM-style ancestral sampling from the **same trained score model**
7. Compare stochastic vs. deterministic samplers + checkpoint

All experiments stay on normalized **2D two-moons data**, so the evolving marginals, score
field, and reverse trajectories remain visible. This follows the same "understand it in 2D first"
philosophy as the previous notebooks.


In [ ]:
# --- Setup -------------------------------------------------------------
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Set True for a quick syntax / plumbing check.
# Leave False for the actual learning experiment (GPU recommended; CPU also works).
FAST_DEV_RUN = False

TRAIN_STEPS = 400 if FAST_DEV_RUN else 6000
SAMPLE_STEPS = 40 if FAST_DEV_RUN else 300
ODE_STEPS = 20 if FAST_DEV_RUN else 75
N_SAMPLES = 400 if FAST_DEV_RUN else 1200


## 4.1 From Discrete Noise Levels to a Continuous-Time SDE

Both NCSN and DDPM can be written, over a tiny time increment $\Delta t$, in the common form
(Ch. 4, Eq. 4.1.1):

$$
x_{t+\Delta t}
\approx x_t + f(x_t,t)\Delta t + g(t)\sqrt{\Delta t}\,\epsilon_t,
\qquad \epsilon_t\sim\mathcal N(0,I).
$$

As $\Delta t\to 0$, this becomes the **forward SDE**

$$
dx(t) = f(x(t),t)\,dt + g(t)\,dw(t),
\qquad x(0)\sim p_{\text{data}}.
$$

The forward process defines a whole family of noisy marginals $p_t$. The object we learn is the
**time-dependent score**

$$
s(x,t) = \nabla_x \log p_t(x).
$$

Once that score is known, Ch. 4 gives two different generation dynamics with the **same marginal
densities**:

**Reverse-time SDE** (Eq. 4.1.6), solved from $T\to 0$:

$$
dx = \Big[f(x,t) - g(t)^2\nabla_x\log p_t(x)\Big]dt + g(t)\,d\bar w(t).
$$

**Probability Flow ODE** (Eq. 4.1.7):

$$
\frac{dx}{dt}
= f(x,t) - \frac12 g(t)^2\nabla_x\log p_t(x).
$$

That factor of $\tfrac12$ matters: the PF-ODE is **not** obtained by simply deleting the noise
term from the reverse SDE. Its drift is chosen so that its density evolution matches the same
$p_t$ family.


In [ ]:
# --- Toy dataset: normalized two moons ---------------------------------
X, _ = make_moons(n_samples=6000, noise=0.05, random_state=0)
X = torch.tensor(X, dtype=torch.float32)
X = (X - X.mean(0)) / X.std(0)  # roughly zero mean, unit variance per coordinate

plt.figure(figsize=(4, 4))
plt.scatter(X[:, 0], X[:, 1], s=4, alpha=0.6)
plt.title("Target distribution: normalized two moons")
plt.axis("equal")
plt.show()

X_dev = X.to(device)


## 4.2 One Forward Kernel, Two Important SDEs

Section 4.5 gives a particularly clean way to think about the forward process. Instead of starting
from infinitesimal transitions, specify the accumulated perturbation kernel directly:

$$
p_t(x_t\mid x_0)
= \mathcal N(x_t;\alpha_t x_0,\sigma_t^2 I),
\qquad
x_t = \alpha_t x_0 + \sigma_t\epsilon,\quad \epsilon\sim\mathcal N(0,I).
$$

For this Gaussian kernel, the **conditional score** used by denoising score matching is closed-form:

$$
\nabla_{x_t}\log p_t(x_t\mid x_0)
= \frac{\alpha_t x_0-x_t}{\sigma_t^2}
= -\frac{\epsilon}{\sigma_t}.
$$

We implement both major Ch. 4 instantiations behind one interface:

| | **VE-SDE** | **VP-SDE** |
|---|---|---|
| Origin | continuous NCSN | continuous DDPM |
| Drift $f(x,t)$ | $0$ | $-\frac12\beta(t)x$ |
| Diffusion $g(t)$ | $\sqrt{d\,\sigma_{\text{sched}}^2(t)/dt}$ | $\sqrt{\beta(t)}$ |
| Signal coefficient $\alpha_t$ | $1$ | $\exp[-\frac12\int_0^t\beta(s)ds]$ |
| Prior | $\mathcal N(0,\sigma_{\max}^2I)$ (approx.) | $\mathcal N(0,I)$ |

For VE, the book's perturbation variance is
$\sigma_{\text{sched}}^2(t)-\sigma_{\text{sched}}^2(0)$, so the helper
`sigma(t)` below returns the **accumulated perturbation standard deviation**, which is zero at
$t=0$.


In [ ]:
# --- Unified SDE interface ----------------------------------------------
class SDE:
    T = 1.0
    eps = 1e-3  # avoid exactly t=0 during score training/sampling

    def alpha(self, t):
        """Signal coefficient in x_t = alpha(t) * x0 + sigma(t) * eps."""
        raise NotImplementedError

    def sigma(self, t):
        """Accumulated perturbation standard deviation."""
        raise NotImplementedError

    def f_coeff(self, t):
        """For the linear drift f(x,t) = f_coeff(t) * x."""
        raise NotImplementedError

    def g(self, t):
        """Scalar diffusion coefficient."""
        raise NotImplementedError

    def drift(self, x, t):
        return self.f_coeff(t).view(-1, 1) * x

    def q_sample(self, x0, t, noise=None):
        """Simulation-free draw from p_t(x_t | x_0)."""
        if noise is None:
            noise = torch.randn_like(x0)
        a = self.alpha(t).view(-1, 1)
        s = self.sigma(t).view(-1, 1)
        return a * x0 + s * noise, noise

    def conditional_score(self, noise, t):
        """Closed-form conditional score = -eps / sigma_t."""
        s = self.sigma(t).view(-1, 1)
        return -noise / s

    def prior_sample(self, n, dim=2, generator=None):
        raise NotImplementedError


class VESDE(SDE):
    """
    Variance Exploding SDE (continuous NCSN).

    Book schedule:
        sigma_sched(t) = sigma_min * (sigma_max / sigma_min)^t
        g(t)^2 = d sigma_sched(t)^2 / dt

    The accumulated perturbation std is:
        sqrt(sigma_sched(t)^2 - sigma_min^2).
    """
    def __init__(self, sigma_min=0.01, sigma_max=5.0):
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max
        self.log_ratio = math.log(sigma_max / sigma_min)

    def sigma_sched(self, t):
        return self.sigma_min * torch.exp(self.log_ratio * t)

    def alpha(self, t):
        return torch.ones_like(t)

    def sigma(self, t):
        var = self.sigma_sched(t) ** 2 - self.sigma_min ** 2
        return torch.sqrt(torch.clamp(var, min=1e-12))

    def f_coeff(self, t):
        return torch.zeros_like(t)

    def g(self, t):
        # sqrt(d sigma_sched^2 / dt)
        return self.sigma_sched(t) * math.sqrt(2.0 * self.log_ratio)

    def prior_sample(self, n, dim=2, generator=None):
        return self.sigma_max * torch.randn(
            n, dim, device=device, generator=generator
        )


class VPSDE(SDE):
    """
    Variance Preserving SDE (continuous DDPM):
        beta(t) = beta_min + t (beta_max - beta_min)
        dx = -1/2 beta(t) x dt + sqrt(beta(t)) dw
    """
    def __init__(self, beta_min=0.1, beta_max=20.0):
        self.beta_min = beta_min
        self.beta_max = beta_max

    def beta(self, t):
        return self.beta_min + t * (self.beta_max - self.beta_min)

    def integral_beta(self, t):
        return (
            self.beta_min * t
            + 0.5 * (self.beta_max - self.beta_min) * t ** 2
        )

    def alpha(self, t):
        return torch.exp(-0.5 * self.integral_beta(t))

    def sigma(self, t):
        # sigma_t^2 = 1 - exp(-integral beta)
        return torch.sqrt(
            torch.clamp(1.0 - torch.exp(-self.integral_beta(t)), min=1e-12)
        )

    def f_coeff(self, t):
        return -0.5 * self.beta(t)

    def g(self, t):
        return torch.sqrt(self.beta(t))

    def prior_sample(self, n, dim=2, generator=None):
        return torch.randn(n, dim, device=device, generator=generator)


ve = VESDE()
vp = VPSDE()

print("VE terminal alpha, sigma:",
      ve.alpha(torch.tensor([1.0])).item(),
      ve.sigma(torch.tensor([1.0])).item())
print("VP terminal alpha, sigma:",
      vp.alpha(torch.tensor([1.0])).item(),
      vp.sigma(torch.tensor([1.0])).item())


In [ ]:
# --- Visualize forward marginals ----------------------------------------
@torch.no_grad()
def plot_forward_marginals(sde, title, times=(0.0, 0.25, 0.5, 0.75, 1.0), n=1500):
    fig, axes = plt.subplots(1, len(times), figsize=(3 * len(times), 3))

    x0 = X_dev[:n]
    shared_noise = torch.randn_like(x0)

    for ax, t_value in zip(axes, times):
        if t_value == 0.0:
            xt = x0
        else:
            t = torch.full((n,), t_value, device=device)
            xt, _ = sde.q_sample(x0, t, shared_noise)

        xt = xt.cpu()
        ax.scatter(xt[:, 0], xt[:, 1], s=3, alpha=0.5)
        ax.set_title(f"t={t_value:.2f}")
        ax.axis("equal")
        ax.set_xticks([])
        ax.set_yticks([])

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_forward_marginals(ve, "VE-SDE forward marginals: variance explodes")
plot_forward_marginals(vp, "VP-SDE forward marginals: signal shrinks, total variance stays bounded")


**What to notice**

- **VE-SDE:** the data are not contracted ($\alpha_t=1$); the cloud simply becomes broader and
  broader until the injected noise dominates the original geometry.
- **VP-SDE:** the signal coefficient $\alpha_t$ decays while the perturbation variance grows toward
  one. The process ends near a standard Gaussian.
- In both cases, training does **not** need to numerically simulate the forward SDE. Because the
  perturbation kernel is Gaussian, we can jump directly from $x_0$ to any randomly sampled $t$.


## 4.3 Continuous-Time Denoising Score Matching

The score $\nabla_x\log p_t(x)$ is unknown because $p_t$ is a marginal over all clean data points.
Exactly as in Ch. 3, conditioning on the known training example $x_0$ turns the target into a
closed-form Gaussian score.

Ch. 4, Eq. 4.3.1:

$$
\mathcal L_{\text{DSM}}(\phi)
=
\frac12\,
\mathbb E_{t,x_0,x_t}
\left[
\omega(t)
\left\|
s_\phi(x_t,t)
-
\nabla_{x_t}\log p_t(x_t\mid x_0)
\right\|_2^2
\right].
$$

For our affine Gaussian kernel,

$$
\nabla_{x_t}\log p_t(x_t\mid x_0)=-\epsilon/\sigma_t.
$$

The minimizer is the *marginal* score
$s^*(x_t,t)=\nabla_{x_t}\log p_t(x_t)$ (Proposition 4.3.1).

We use $\omega(t)=\sigma_t^2$. This is the same scale-balancing idea used in the NCSN notebook:
the target magnitude grows like $1/\sigma_t$, so multiplying squared error by $\sigma_t^2$ keeps
different times on a more comparable scale.


In [ ]:
# --- Time-conditioned score network -------------------------------------
class SinusoidalTimeEmbed(nn.Module):
    """
    Continuous-time version of the timestep embedding used in the DDPM notebook.
    t lives in [0,1]; multiplying by 1000 exposes a useful range of frequencies.
    """
    def __init__(self, dim=32):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000)
            * torch.arange(half, device=t.device)
            / half
        )
        args = (1000.0 * t[:, None]) * freqs[None]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class TimeScoreNet(nn.Module):
    """s_phi(x_t, t): R^2 x [0,1] -> R^2."""
    def __init__(self, dim=2, hidden=128, t_dim=32):
        super().__init__()
        self.temb = SinusoidalTimeEmbed(t_dim)
        self.net = nn.Sequential(
            nn.Linear(dim + t_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, dim),
        )

    def forward(self, x, t):
        return self.net(torch.cat([x, self.temb(t)], dim=-1))


In [ ]:
# --- Generic continuous-time DSM training loop --------------------------
def train_score_sde(model, sde, data, n_steps=8000, batch_size=256, lr=2e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for step in range(n_steps):
        idx = torch.randint(0, data.size(0), (batch_size,), device=device)
        x0 = data[idx]

        # Uniform t in (eps, T]. Avoid exactly t=0 because sigma(0)=0.
        t = sde.eps + (sde.T - sde.eps) * torch.rand(batch_size, device=device)

        # Simulation-free x_t ~ p_t(x_t | x0)
        xt, noise = sde.q_sample(x0, t)

        # Closed-form conditional score target: -eps / sigma_t
        target = sde.conditional_score(noise, t)
        pred = model(xt, t)

        # omega(t) = sigma_t^2
        weight = sde.sigma(t) ** 2
        loss = (weight * ((pred - target) ** 2).sum(dim=1)).mean()

        opt.zero_grad()
        loss.backward()
        opt.step()

        if step % max(1, n_steps // 8) == 0:
            print(f"step {step:5d} | loss {loss.item():.4f}")

    return model


In [ ]:
# --- Train a VE score model (continuous NCSN) ---------------------------
torch.manual_seed(1)
score_ve = TimeScoreNet().to(device)
score_ve = train_score_sde(score_ve, ve, X_dev, n_steps=TRAIN_STEPS)


In [ ]:
# --- Train a VP score model (continuous DDPM) ---------------------------
torch.manual_seed(2)
score_vp = TimeScoreNet().to(device)
score_vp = train_score_sde(score_vp, vp, X_dev, n_steps=TRAIN_STEPS)


In [ ]:
# --- Visualize the learned time-dependent score fields ------------------
@torch.no_grad()
def plot_score_field(model, t_value, ax, xlim=(-2.5, 2.5), ylim=(-2.5, 2.5), grid=18):
    xs = torch.linspace(*xlim, grid, device=device)
    ys = torch.linspace(*ylim, grid, device=device)
    xx, yy = torch.meshgrid(xs, ys, indexing="xy")
    pts = torch.stack([xx.flatten(), yy.flatten()], dim=1)
    t = torch.full((pts.shape[0],), t_value, device=device)

    score = model(pts, t).cpu()
    pts = pts.cpu()

    ax.quiver(pts[:, 0], pts[:, 1], score[:, 0], score[:, 1], alpha=0.55)
    ax.set_title(f"t={t_value:.2f}")
    ax.axis("equal")


fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, t_value in zip(axes[0], [0.05, 0.4, 0.9]):
    plot_score_field(score_ve, t_value, ax)
axes[0, 0].set_ylabel("VE score field")

for ax, t_value in zip(axes[1], [0.05, 0.4, 0.9]):
    plot_score_field(score_vp, t_value, ax)
axes[1, 0].set_ylabel("VP score field")

plt.tight_layout()
plt.show()


At small $t$, the score field has to resolve the sharp local geometry of the two moons. At large
$t$, the noisy marginal is smoother and the field becomes much more global. This is the
continuous-time version of the coarse-to-fine score fields from NCSN — except now the network is
trained over a **continuum** of noise levels rather than a finite ladder.


## 4.4 Reverse-Time SDE Sampling — Euler–Maruyama

After training, replace the oracle score in the reverse SDE by $s_{\phi^\times}(x,t)$:

$$
dx
=
\left[
f(x,t)-g(t)^2s_{\phi^\times}(x,t)
\right]dt
+
g(t)d\bar w(t).
$$

Ch. 4, Eq. 4.3.4 applies **Euler–Maruyama** while integrating backward in time.
If our grid has $t_{\text{next}}<t$, define $h=t_{\text{next}}-t<0$:

$$
x_{\text{next}}
=
x
+
\left[f(x,t)-g(t)^2s_\phi(x,t)\right]h
+
g(t)\sqrt{-h}\,\epsilon.
$$

The drift uses the score to pull samples toward the evolving high-density region; the Brownian term
keeps the reverse process stochastic.


In [ ]:
# --- Reverse-time SDE sampler (generic VE or VP) ------------------------
@torch.no_grad()
def reverse_sde_sample(
    model,
    sde,
    n_samples=2000,
    dim=2,
    n_steps=500,
    initial=None,
    seed=0,
    record_every=None,
):
    generator = torch.Generator(device=device).manual_seed(seed)

    if initial is None:
        x = sde.prior_sample(n_samples, dim, generator=generator)
    else:
        x = initial.to(device).clone()
        n_samples = x.shape[0]

    times = torch.linspace(sde.T, sde.eps, n_steps + 1, device=device)
    traj = [x.cpu().clone()] if record_every else None

    for i in range(n_steps):
        t_now = times[i]
        t_next = times[i + 1]
        h = t_next - t_now  # negative: we integrate T -> 0

        t = torch.full((n_samples,), float(t_now), device=device)
        score = model(x, t)
        g = sde.g(t).view(-1, 1)

        reverse_drift = sde.drift(x, t) - g ** 2 * score
        noise = torch.randn(
            x.shape, device=device, generator=generator, dtype=x.dtype
        )

        x = x + reverse_drift * h + g * torch.sqrt(-h) * noise

        if record_every and (i + 1) % record_every == 0:
            traj.append(x.cpu().clone())

    return x.cpu(), traj


In [ ]:
# --- Sample VE and VP with the same generic reverse-SDE solver ----------
samples_ve_sde, traj_ve = reverse_sde_sample(
    score_ve, ve,
    n_samples=N_SAMPLES,
    n_steps=SAMPLE_STEPS,
    seed=10,
    record_every=max(1, SAMPLE_STEPS // 5),
)

samples_vp_sde, traj_vp = reverse_sde_sample(
    score_vp, vp,
    n_samples=N_SAMPLES,
    n_steps=SAMPLE_STEPS,
    seed=11,
    record_every=max(1, SAMPLE_STEPS // 5),
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].scatter(X[:, 0], X[:, 1], s=3, alpha=0.5)
axes[0].set_title("Data")
axes[1].scatter(samples_ve_sde[:, 0], samples_ve_sde[:, 1], s=3, alpha=0.5)
axes[1].set_title("VE reverse SDE")
axes[2].scatter(samples_vp_sde[:, 0], samples_vp_sde[:, 1], s=3, alpha=0.5)
axes[2].set_title("VP reverse SDE")
for ax in axes:
    ax.axis("equal")
plt.tight_layout()
plt.show()


In [ ]:
# --- Watch the VP reverse SDE turn noise into data ----------------------
fig, axes = plt.subplots(1, len(traj_vp), figsize=(3 * len(traj_vp), 3))
for ax, snap in zip(axes, traj_vp):
    ax.scatter(snap[:, 0], snap[:, 1], s=2, alpha=0.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.axis("equal")
plt.suptitle("VP reverse SDE: prior -> data")
plt.tight_layout()
plt.show()


## 4.5 Probability Flow ODE — Same Marginals, Deterministic Paths

The PF-ODE associated with the same forward SDE is

$$
\frac{dx}{dt}
=
f(x,t)
-
\frac12 g(t)^2 s_{\phi^\times}(x,t).
$$

Unlike the reverse SDE, this dynamics has **no Brownian term**, so a fixed initial latent produces
a fixed trajectory. Yet the Fokker–Planck / continuity-equation argument in Section 4.2 shows that
the PF-ODE is constructed to share the same marginal family $p_t$ as the forward/reverse SDEs.

Euler works, but an ODE lets us use standard higher-order solvers. Below we implement
**RK4 from scratch** to avoid an extra dependency.


In [ ]:
# --- PF-ODE + RK4 solver ------------------------------------------------
@torch.no_grad()
def probability_flow_drift(model, sde, x, t_value):
    t = torch.full((x.shape[0],), float(t_value), device=x.device)
    g = sde.g(t).view(-1, 1)
    score = model(x, t)
    return sde.drift(x, t) - 0.5 * g ** 2 * score


@torch.no_grad()
def pf_ode_sample(
    model,
    sde,
    n_samples=2000,
    dim=2,
    n_steps=125,
    initial=None,
    seed=0,
    record_every=None,
):
    generator = torch.Generator(device=device).manual_seed(seed)

    if initial is None:
        x = sde.prior_sample(n_samples, dim, generator=generator)
    else:
        x = initial.to(device).clone()
        n_samples = x.shape[0]

    times = torch.linspace(sde.T, sde.eps, n_steps + 1, device=device)
    traj = [x.cpu().clone()] if record_every else None

    for i in range(n_steps):
        t = times[i]
        t_next = times[i + 1]
        h = t_next - t  # negative

        k1 = probability_flow_drift(model, sde, x, t)
        k2 = probability_flow_drift(model, sde, x + 0.5 * h * k1, t + 0.5 * h)
        k3 = probability_flow_drift(model, sde, x + 0.5 * h * k2, t + 0.5 * h)
        k4 = probability_flow_drift(model, sde, x + h * k3, t_next)

        x = x + (h / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)

        if record_every and (i + 1) % record_every == 0:
            traj.append(x.cpu().clone())

    return x.cpu(), traj


In [ ]:
# --- Sample with PF-ODE --------------------------------------------------
samples_ve_ode, _ = pf_ode_sample(
    score_ve, ve,
    n_samples=N_SAMPLES,
    n_steps=ODE_STEPS,
    seed=20,
)

samples_vp_ode, traj_vp_ode = pf_ode_sample(
    score_vp, vp,
    n_samples=N_SAMPLES,
    n_steps=ODE_STEPS,
    seed=21,
    record_every=max(1, ODE_STEPS // 5),
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].scatter(X[:, 0], X[:, 1], s=3, alpha=0.5)
axes[0].set_title("Data")
axes[1].scatter(samples_ve_ode[:, 0], samples_ve_ode[:, 1], s=3, alpha=0.5)
axes[1].set_title("VE PF-ODE (RK4)")
axes[2].scatter(samples_vp_ode[:, 0], samples_vp_ode[:, 1], s=3, alpha=0.5)
axes[2].set_title("VP PF-ODE (RK4)")
for ax in axes:
    ax.axis("equal")
plt.tight_layout()
plt.show()

print("Reverse SDE score evaluations (approx.):", SAMPLE_STEPS)
print("PF-ODE RK4 score evaluations (approx.):", 4 * ODE_STEPS)


**What to notice:** the SDE and PF-ODE do not trace the same *individual* path. The reverse SDE
keeps injecting calibrated randomness, while the PF-ODE transports points deterministically.
The important Ch. 4 statement is distributional: with the **oracle** score and exact integration,
both reproduce the same family of marginals $p_t$. With a learned score and finite solver steps,
small visual differences are expected because of model error and numerical discretization error.


## 4.6 VP-SDE as Continuous DDPM — A Third Sampler

The learning-path checkpoint asks for the **same trained model** to support three samplers:
ancestral, reverse SDE, and PF-ODE.

Section 4.5 makes this possible without training a separate DDPM noise-predictor. For the affine
kernel

$$
x_s = \alpha_s x_0 + \sigma_s\epsilon,
$$

the learned marginal score implies the posterior-mean clean prediction (generalized Tweedie):

$$
\hat x_0(x_s,s)
=
\frac{x_s+\sigma_s^2 s_\phi(x_s,s)}{\alpha_s}.
$$

For two times $0\le t<s$, Lemma 4.5.2 gives the Gaussian reverse conditional

$$
p(x_t\mid x_s,x_0)
=
\mathcal N\left(x_t;\mu(x_s,x_0;s,t),\sigma^2(s,t)I\right).
$$

Replacing $x_0$ by $\hat x_0$ gives a DDPM-style **ancestral** sampler. This is a discrete
reverse chain; the reverse-SDE sampler is its continuous-time limit under the VP dynamics.


In [ ]:
# --- VP ancestral sampler using the SAME score_vp model -----------------
@torch.no_grad()
def vp_ancestral_sample(
    model,
    sde,
    n_samples=2000,
    dim=2,
    n_steps=500,
    initial=None,
    seed=0,
):
    assert isinstance(sde, VPSDE), "This demo is written for the VP/DDPM case."

    generator = torch.Generator(device=device).manual_seed(seed)

    if initial is None:
        x = sde.prior_sample(n_samples, dim, generator=generator)
    else:
        x = initial.to(device).clone()
        n_samples = x.shape[0]

    # Current times s descend from T to eps. Each step targets the next earlier t.
    current_times = torch.linspace(sde.T, sde.eps, n_steps, device=device)
    next_times = torch.cat([
        current_times[1:],
        torch.tensor([0.0], device=device),
    ])

    for s_value, t_value in zip(current_times, next_times):
        s = torch.full((n_samples,), float(s_value), device=device)
        alpha_s = sde.alpha(s).view(-1, 1)
        sigma_s = sde.sigma(s).view(-1, 1)

        # E[x0 | x_s] implied by the learned marginal score.
        score = model(x, s)
        x0_hat = (x + sigma_s ** 2 * score) / alpha_s

        # Final step goes straight to the denoised prediction at t=0.
        if float(t_value) == 0.0:
            x = x0_hat
            break

        t = torch.full((n_samples,), float(t_value), device=device)
        alpha_t = sde.alpha(t).view(-1, 1)
        sigma_t = sde.sigma(t).view(-1, 1)

        # Lemma 4.5.2 notation:
        # alpha_{s|t} = alpha_s / alpha_t
        # sigma_{s|t}^2 = sigma_s^2 - alpha_{s|t}^2 sigma_t^2
        alpha_s_given_t = alpha_s / alpha_t
        sigma_s_given_t_sq = (
            sigma_s ** 2
            - alpha_s_given_t ** 2 * sigma_t ** 2
        )
        sigma_s_given_t_sq = torch.clamp(sigma_s_given_t_sq, min=1e-12)

        mean = (
            alpha_s_given_t * (sigma_t ** 2 / sigma_s ** 2) * x
            + alpha_t * (sigma_s_given_t_sq / sigma_s ** 2) * x0_hat
        )

        variance = sigma_s_given_t_sq * sigma_t ** 2 / sigma_s ** 2
        noise = torch.randn(
            x.shape, device=device, generator=generator, dtype=x.dtype
        )
        x = mean + torch.sqrt(torch.clamp(variance, min=0.0)) * noise

    return x.cpu()


In [ ]:
# --- Same VP score model, three samplers --------------------------------
# Use the same starting latent for all three so the difference is sampler-only.
latent_gen = torch.Generator(device=device).manual_seed(123)
shared_latent = vp.prior_sample(N_SAMPLES, 2, generator=latent_gen)

samples_vp_ancestral = vp_ancestral_sample(
    score_vp, vp,
    n_steps=SAMPLE_STEPS,
    initial=shared_latent,
    seed=30,
)

samples_vp_sde_same, _ = reverse_sde_sample(
    score_vp, vp,
    n_steps=SAMPLE_STEPS,
    initial=shared_latent,
    seed=31,
)

samples_vp_ode_same, _ = pf_ode_sample(
    score_vp, vp,
    n_steps=ODE_STEPS,
    initial=shared_latent,
    seed=32,  # irrelevant once `initial` is fixed; ODE itself is deterministic
)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
plots = [
    ("Data", X),
    ("VP ancestral", samples_vp_ancestral),
    ("VP reverse SDE", samples_vp_sde_same),
    ("VP PF-ODE", samples_vp_ode_same),
]
for ax, (title, pts) in zip(axes, plots):
    ax.scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.5)
    ax.set_title(title)
    ax.axis("equal")
plt.tight_layout()
plt.show()


In [ ]:
# --- Determinism check: SDE changes, PF-ODE does not --------------------
small_latent = shared_latent[:300]

sde_run_1, _ = reverse_sde_sample(
    score_vp, vp,
    n_steps=SAMPLE_STEPS,
    initial=small_latent,
    seed=100,
)
sde_run_2, _ = reverse_sde_sample(
    score_vp, vp,
    n_steps=SAMPLE_STEPS,
    initial=small_latent,
    seed=101,
)

ode_run_1, _ = pf_ode_sample(
    score_vp, vp,
    n_steps=ODE_STEPS,
    initial=small_latent,
)
ode_run_2, _ = pf_ode_sample(
    score_vp, vp,
    n_steps=ODE_STEPS,
    initial=small_latent,
)

print("Mean |SDE run 1 - SDE run 2|:",
      (sde_run_1 - sde_run_2).abs().mean().item())
print("Max  |ODE run 1 - ODE run 2|:",
      (ode_run_1 - ode_run_2).abs().max().item())


## 4.7 Why the Marginals Match: Fokker–Planck in One Line

For a scalar-diffusion forward SDE

$$
dx = f(x,t)\,dt + g(t)\,dw,
$$

the marginal density satisfies the **Fokker–Planck equation**

$$
\partial_t p_t(x)
=
-\nabla\cdot\big(f(x,t)p_t(x)\big)
+
\frac12 g(t)^2\Delta p_t(x).
$$

The reverse-time SDE and PF-ODE are engineered so their density evolution reproduces this same
$p_t$ family. That is the unifying principle behind the apparent coincidence:

- VE-SDE and VP-SDE can use the same training recipe.
- The reverse SDE and PF-ODE can use the same learned score.
- DDPM ancestral sampling appears as a discretized VP reverse process.
- NCSN appears as a discretized VE process.

So the conceptual jump in Ch. 4 is not "a new diffusion model." It is a **continuous-time language**
in which the models from Ch. 2 and Ch. 3 become special cases and sampling becomes a numerical
differential-equation problem.


## 8. Exercises / Checkpoint

1. **VE vs. VP forward process.** Change `sigma_max` in `VESDE` and `beta_max` in `VPSDE`.
   Which parameter has to be large enough for $p_T$ to become essentially independent of $x_0$?
2. **Discretization error.** Reduce `SAMPLE_STEPS` from 300 to 50. Which degrades faster:
   the ancestral sampler or Euler–Maruyama reverse SDE?
3. **ODE order.** Replace RK4 with a one-line Euler PF-ODE update. At the same number of steps,
   which solver preserves the two-moons geometry better?
4. **Stochasticity.** Reuse one fixed `shared_latent` and run the reverse SDE with five different
   random seeds. Then do the same with PF-ODE. Explain the difference.
5. **Score ↔ noise prediction.** For the VP model, verify numerically that
   $\hat\epsilon(x_t,t)=-\sigma_t s_\phi(x_t,t)$ and use it to rewrite the ancestral sampler in
   DDPM-style $\epsilon$-prediction notation.
6. **Optional theory.** Starting from the VP perturbation kernel, differentiate
   $\alpha_t=\exp[-\frac12\int_0^t\beta(s)ds]$ and recover
   $f(x,t)=-\frac12\beta(t)x$.

**Checkpoint:** with the **same trained VP score network**, the ancestral sampler, reverse-time SDE,
and PF-ODE should all turn a Gaussian prior into visually consistent two-moons samples. You should
also be able to explain why the SDE is stochastic, why the PF-ODE is deterministic, and why both
can still agree at the level of marginal distributions.
